# SOURCE

## Idea clave
Cuándo se ejecuta un script normal:
```bash
./script.sh
```
  * Bash crea **un nuevo proceso (subshell)**
  * Todo lo que pase ahí **no afecta tu shell actual**

En cambio, con:
```bash
source script.sh
# O su equivalente
. script.sh

```

  * El script se ejecuta en el **mismo shell actual**
  * Variables, funciones y cambios **se quedan disponibles**

### Ejemplo básico

#### Archivo: `config.sh`
```bash
NOMBRE="Luis"
EDAD=24
```

---

#### Caso 1: Ejecución normal
```bash
bash config.sh
echo $NOMBRE
```

Resultado:
```bash
(no imprime nada)
```
> Porque la variable murió en el subshell.

#### Caso 2: Usando `source`
```bash
source config.sh
echo $NOMBRE
```
Resultado:
```bash
Luis
```
> Aquí sí funciona porque la variable se cargó en tu shell actual.




## Cuándo usar `source` (regla práctica)

Usa `source` cuando quieras:
  * ✔ Compartir variables
  * ✔ Reutilizar funciones
  * ✔ Cargar configuración
  * ✔ Modificar el entorno actual

NO lo uses cuando:
  * Solo quieres ejecutar un script aislado

### Ejemplo 1

#### Archivo: `config.sh`

```bash
RUTA_BACKUP="/home/user/backups"
USUARIO="admin"
```

---

#### Archivo: `backup.sh`

```bash
#!/bin/bash

source config.sh

echo "Guardando backup en: $RUTA_BACKUP"
echo "Usuario: $USUARIO"
```

---

Ventajas:
  * Separas lógica y configuración
  * Puedes cambiar valores sin tocar el script principal

### Ejemplo 2

#### Archivo: `utils.sh`

```bash
saludar() {
  echo "Hola, $1"
}
```

---

#### Archivo: `main.sh`

```bash
#!/bin/bash

source utils.sh

saludar "Luis"
```

Resultado:

```
Hola, Luis
```
> Sin `source`, la función no existiría en `main.sh`.

## `BASH_SOURCE`
Es una **variable especial de bash** (un array).

Es un array que bash llena automáticamente. En cada momento de la ejecución, contiene la pila de archivos que están siendo ejecutados o sourced:
```bash
BASH_SOURCE[0]  # archivo actual (donde está esta línea)
BASH_SOURCE[1]  # quién llamó al archivo actual
BASH_SOURCE[2]  # quién llamó a ese, etc.
```
Acompañado siempre de `FUNCNAME` y `BASH_LINENO` — los tres juntos forman el stack trace completo.

### Ejemplo básico
```bash
# archivo: test.sh
echo "Estoy en: ${BASH_SOURCE[0]}"
```
Ouput:
```ouput
Estoy en: test.sh
```

---

### Ejemplo con `source`
```bash
# archivo: lib.sh
echo "lib.sh llamado desde: ${BASH_SOURCE[0]}"
```
> `${BASH_SOURCE[0]}`Devuelve la ruta del script actual

```bash
#archivo: main sh
source lib.sh
```
Ouput:
```ouput
lib.sh llamado desde: lib.sh
```

---

### ¿Para qué sirve?
Para debugging avanzado

> Saber donde estás parado
```bash
echo "Archivo actual:"
```

#### Ejemplo más claro
```bash
# lib.sh

debug() {
  echo "Archivo actual: ${BASH_SOURCE[0]}"
  echo "Quién me llamó: ${BASH_SOURCE[1]}"
}
```
```bash
# main.sh

source lib.sh
debug
```
Ouput:
```bash
Archivo actual: lib.sh
Quién me llamó: main.sh
```

---

### Mini Framework | Organizar scritps
La idea de "framework" en bash No es algo gigante.

Es simplemente:
  * separar archivos (`config`, `funciones`, `main`)
  * cargarlos con `source`
  * tener un orden claro

#### Estructura básica
```bash
project/
├── main.sh
├── lib.sh
└── config.sh
```

#### `config.sh`
```bash
APP_NAME="MiApp"
VERSION="1.0"
```

#### `lib.sh`
```bash
log() {
    echo "[LOG] $1"
}
```

#### `main.sh`
```bash
source config.sh
source lib.sh

log "Iniciando $APP_NAME v$VERSION"
```
Ejecutas:
```bash
bash main.sh
```

> Ya se hizo un framework pequeño

#### Problema real
Si se ejecuta `main` desde otra carpeta.
```bash
cd /
bash /ruta/a/proyecto/main.sh
```
Puede romperse:
```bash
source config.sh
```

Porque ya no está en el mismo directorio.

---

#### Aquí entra `BASH_SOURCE`
Solución real:
```bash
SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
```
Traduccion humana:
  * `${BASH_SOURCE[0]}`-> archivo actual
  * `dirname` -> su carpeta
  * `cd && pwd` → ruta absoluta

##### `main.sh` mejorado
```bash
SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")"&& pwd)"

source "$SCRIPT_DIR/config.sh"
source "$SCRIPT_DIR/config.sh"

log "Iniciando $APP_NAME v$VERSION"
```
> Ahora funciona desde cualquier lado


---




### EXTRA

#### LINENO
`LINENO` te dice:

> en qué línea del archivo estás ejecutando código.

```bash
echo "Línea: $LINENO"
echo "Línea: $LINENO" 
```
Ouput:
```ouput
Línea: 1
Línea: 2
```
Dentro de funciones puede ser medio confuso:
```bash
mi_func() {
    echo "Estoy en línea: $LINENO"
}
```
> `LINENO` sigue contando líneas del archivo, no “líneas dentro de la función”.

Hay `BASH_LINENO` también es un array que guarda **las líneas desde donde se llamaron funciones**

| Variable      | Qué dice                    |
| ------------- | --------------------------- |
| `LINENO`      | dónde estás parado ahora    |
| `BASH_LINENO` | desde qué línea te llamaron |

#### `FUNCNAME`
Quién esta ejecutando.

> Es un array que guarda la “pila de llamadas” (call stack).

```bash
mi_func() {
    echo "Función actual: ${FUNCNAME[0]}"
}

mi_func
```
Ouput:
```ouput
Función actual: mi_func
```

#### Ejemplo
```bash
a() {
    b
}

b() {
    c
}

c() {
    echo "Actual: ${FUNCNAME[0]}"
    echo "Quién llamó: ${FUNCNAME[1]}"
    echo "Más arriba: ${FUNCNAME[2]}"
}

a
```
Ouput:
```ouput
Actual: c
Quién llamó: b
Más arriba: a
```

---

#### Stack trace
Ver quién llamó a quién

##### `lib.sh`
```bash
trace() {
    echo "--- TRACE ---"
    for i in "${!BASH_SOURCE[@]}"
    do
      echo "$i -> ${BASH_SOURCE[$i]}"
    done
}
```

##### `main.sh`
```bash
source lib.sh

func_a() {
    func_b
}

func_b() {
    trace
}

func_a
```
Ouput:
```bash
--- TRACE ---
0 -> lib.sh
1 -> main.sh
2 -> main.sh
3 -> main.sh
```

##### Desglose
| Índice | Qué es realmente                     | Archivo |
| ------ | ------------------------------------ | ------- |
| 0      | `trace` (definida en lib.sh)         | lib.sh  |
| 1      | llamada desde `func_b`               | main.sh |
| 2      | llamada desde `func_a`               | main.sh |
| 3      | ejecución global del script (`main`) | main.sh |

##### Muestra
```bash
trace() {
  echo "---- TRACE ----"
  for i in "${!FUNCNAME[@]}"; do
    echo "$i -> func=${FUNCNAME[$i]} file=${BASH_SOURCE[$i]}"
  done
}
```
Ouput:
```bash
0 -> func=trace file=lib.sh
1 -> func=func_b file=main.sh
2 -> func=func_a file=main.sh
3 -> func=main   file=main.sh
```
El último nivel `func=main` no lo escribes tú, lo pone Bash representa el script ejecutándose.

##### Regla de oro:
Siempre que use:
  * `FUNCNAME`
  * `BASH_SOURCE`
  * `BASH_LINENO`

> el último nivel será el script (main)

---

#### Mini logger
```bash
log() {
    echo "[${BASH_SOURCE[1]}:${FUNCNAME[1]}:${BASH_LINENO[0]}] $1"
}
```

Uso:
```bash
mi_func() {
  log "Hola"
}

mi_func
```
Ouput:
```ouput
[main.sh:mi_func:10] Hola
```